In [14]:
# === CMNIST EVALUATION NOTEBOOK: SETUP ===
import os
import gc

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn.functional as F

import matplotlib.pyplot as plt
import torchvision.transforms as T

print("✅ Basic eval imports ready.")

# You should already have loaded:
# - all_results
# - cv_folds
# - omega
# - Dll_samples, Dhl_samples
# - det_ll_dict, det_ll_dict_opt, det_hl_dict_opt
# - U_ll_hat_fixed, U_hl_hat_fixed, U_ll_hat_opt, U_hl_hat_opt

# Optional: set default output dir for eval results
output_dir = 'data/cmnist/results_empirical'
os.makedirs(output_dir, exist_ok=True)
print(f"Results will be saved under: {output_dir}")


✅ Basic eval imports ready.
Results will be saved under: data/cmnist/results_empirical


In [15]:
# === Cell 0: Load training results and CMNIST structures ===
import os
import pickle
import torch

print("Loading CMNIST eval state...")

# TODO: adapt these paths to your actual saved files
base_dir = "data/cmnist/results_empirical"

# Example: merged optimization results (all_results)
all_results_path = os.path.join(base_dir, "cmnist_all_results.pkl")
if os.path.exists(all_results_path):
    with open(all_results_path, "rb") as f:
        all_results = pickle.load(f)
    print(f"✓ Loaded all_results from {all_results_path}")
else:
    print(f"✗ Could not find all_results at {all_results_path}. Please update the path.")

# Example: cross-validation folds
cv_folds_path = os.path.join(base_dir, "cmnist_cv_folds.pkl")
if os.path.exists(cv_folds_path):
    with open(cv_folds_path, "rb") as f:
        cv_folds = pickle.load(f)
    print(f"✓ Loaded cv_folds from {cv_folds_path}")
else:
    print(f"✗ Could not find cv_folds at {cv_folds_path}. Please update the path.")

# Example: omega (environment pair mapping)
omega_path = os.path.join(base_dir, "cmnist_omega.pkl")
if os.path.exists(omega_path):
    with open(omega_path, "rb") as f:
        omega = pickle.load(f)
    print(f"✓ Loaded omega from {omega_path}")
else:
    print(f"✗ Could not find omega at {omega_path}. Please update the path.")

# Example: deterministic dicts and noise (opt view)
det_ll_opt_path = os.path.join(base_dir, "cmnist_det_ll_dict_opt.pt")
det_hl_opt_path = os.path.join(base_dir, "cmnist_det_hl_dict_opt.pt")
U_ll_opt_path   = os.path.join(base_dir, "cmnist_U_ll_hat_opt.pt")
U_hl_opt_path   = os.path.join(base_dir, "cmnist_U_hl_hat_opt.pt")

if os.path.exists(det_ll_opt_path):
    det_ll_dict_opt = torch.load(det_ll_opt_path)
    print(f"✓ Loaded det_ll_dict_opt from {det_ll_opt_path}")
else:
    print(f"✗ det_ll_dict_opt not found at {det_ll_opt_path}")

if os.path.exists(det_hl_opt_path):
    det_hl_dict_opt = torch.load(det_hl_opt_path)
    print(f"✓ Loaded det_hl_dict_opt from {det_hl_opt_path}")
else:
    print(f"✗ det_hl_dict_opt not found at {det_hl_opt_path}")

if os.path.exists(U_ll_opt_path):
    U_ll_hat_opt = torch.load(U_ll_opt_path)
    print(f"✓ Loaded U_ll_hat_opt from {U_ll_opt_path}")
else:
    print(f"✗ U_ll_hat_opt not found at {U_ll_opt_path}")

if os.path.exists(U_hl_opt_path):
    U_hl_hat_opt = torch.load(U_hl_opt_path)
    print(f"✓ Loaded U_hl_hat_opt from {U_hl_opt_path}")
else:
    print(f"✗ U_hl_hat_opt not found at {U_hl_opt_path}")

# Example: Dll_samples / Dhl_samples for evaluation
dll_samples_path = os.path.join(base_dir, "cmnist_Dll_samples.pkl")
dhl_samples_path = os.path.join(base_dir, "cmnist_Dhl_samples.pkl")

if os.path.exists(dll_samples_path):
    with open(dll_samples_path, "rb") as f:
        Dll_samples = pickle.load(f)
    print(f"✓ Loaded Dll_samples from {dll_samples_path}")
else:
    print(f"✗ Dll_samples not found at {dll_samples_path}")

if os.path.exists(dhl_samples_path):
    with open(dhl_samples_path, "rb") as f:
        Dhl_samples = pickle.load(f)
    print(f"✓ Loaded Dhl_samples from {dhl_samples_path}")
else:
    print(f"✗ Dhl_samples not found at {dhl_samples_path}")

print("Done loading. Adjust any paths above as needed.")


Loading CMNIST eval state...
✗ Could not find all_results at data/cmnist/results_empirical/cmnist_all_results.pkl. Please update the path.
✗ Could not find cv_folds at data/cmnist/results_empirical/cmnist_cv_folds.pkl. Please update the path.
✗ Could not find omega at data/cmnist/results_empirical/cmnist_omega.pkl. Please update the path.
✗ det_ll_dict_opt not found at data/cmnist/results_empirical/cmnist_det_ll_dict_opt.pt
✗ det_hl_dict_opt not found at data/cmnist/results_empirical/cmnist_det_hl_dict_opt.pt
✗ U_ll_hat_opt not found at data/cmnist/results_empirical/cmnist_U_ll_hat_opt.pt
✗ U_hl_hat_opt not found at data/cmnist/results_empirical/cmnist_U_hl_hat_opt.pt
✗ Dll_samples not found at data/cmnist/results_empirical/cmnist_Dll_samples.pkl
✗ Dhl_samples not found at data/cmnist/results_empirical/cmnist_Dhl_samples.pkl
Done loading. Adjust any paths above as needed.


In [16]:
# === Visualization: adversarial perturbations (pixels-only, DiRoCA) ===
import matplotlib.pyplot as plt
import torch
import numpy as np

print("\n" + "="*20 + " Visualizing Adversarial Perturbations (UPDATED) " + "="*20)

# ------------------------------------------------------------
# 0) Helper: find observational LL key in det_ll_dict
# ------------------------------------------------------------
def find_obs_ll_key(det_ll_dict):
    keys = list(det_ll_dict.keys())
    if not keys:
        raise ValueError("det_ll_dict is empty; cannot find observational key.")
    for k in keys:
        if isinstance(k, str) and k.lower() in ["obs", "observational", "none", "null"]:
            return k
        label = getattr(k, "label", None)
        if isinstance(label, str) and label.lower() in ["obs", "observational"]:
            return k
    print("[Warning] No explicit obs key found. Falling back to first key:", keys[0])
    return keys[0]

# ------------------------------------------------------------
# 1) Select DiRoCA run and fold
# ------------------------------------------------------------
fold_key_to_show = "fold_0"

diroca_key_toplevel = next((k for k in all_results if k.startswith("diroca_eps_")), None)
if diroca_key_toplevel is None:
    print("Error: No DiRoCA results found in all_results.")
    run_result = None
else:
    print(f"Using DiRoCA results: {diroca_key_toplevel}")
    fold_block = all_results[diroca_key_toplevel].get(fold_key_to_show, None)

    if fold_block is None:
        print(f"Error: Fold {fold_key_to_show} not found in {diroca_key_toplevel}")
        run_result = None
    else:
        run_keys = [k for k in fold_block.keys() if k.startswith("eps_")]
        if not run_keys:
            print(f"Error: no inner keys 'eps_*' in fold data. Keys: {list(fold_block.keys())}")
            run_result = None
        else:
            run_key = run_keys[0]
            run_result = fold_block[run_key]
            if isinstance(run_result, dict) and "error" in run_result:
                print(f"Error in run result: {run_result['error']}")
                run_result = None
            else:
                print(f"Using run key: {run_key}")

if run_result is None:
    print("✗ Could not access DiRoCA run_result for visualization.")
else:
    # ------------------------------------------------------------
    # 2) Extract Theta and fold/train indices
    # ------------------------------------------------------------
    final_Theta_ll = run_result.get("final_Theta_ll", None)
    if final_Theta_ll is None:
        raise ValueError("Theta (final_Theta_ll) not found in run_result.")

    if not isinstance(final_Theta_ll, torch.Tensor):
        final_Theta_ll = torch.tensor(final_Theta_ll, dtype=torch.float32)

    epsilon_run = run_result.get("epsilon", "Unknown")
    fold_index = int(fold_key_to_show.split("_")[-1])
    train_indices = cv_folds[fold_index]["train"]

    # ------------------------------------------------------------
    # 3) Get the *exact* U_ll used in training (pixels only)
    # ------------------------------------------------------------
    if "U_ll_hat_fixed" in globals():
        U_ll_base = torch.as_tensor(U_ll_hat_fixed, dtype=torch.float32)
    else:
        U_ll_base = torch.as_tensor(U_ll_hat, dtype=torch.float32)
        if U_ll_base.ndim > 2:
            U_ll_base = U_ll_base.view(U_ll_base.shape[0], -1)

    U_ll_train = U_ll_base[train_indices]  # (N_train, 3072)

    # ------------------------------------------------------------
    # 4) Deterministic LL part for obs key
    # ------------------------------------------------------------
    obs_ll_key = find_obs_ll_key(det_ll_dict)
    det_ll_train_obs = det_ll_dict[obs_ll_key][train_indices]

    if not isinstance(det_ll_train_obs, torch.Tensor):
        det_ll_train_obs = torch.as_tensor(det_ll_train_obs, dtype=torch.float32)

    if det_ll_train_obs.shape[1] == 3092:
        det_pixels_train = det_ll_train_obs[:, :3072]   # (N_train, 3072)
    elif det_ll_train_obs.shape[1] == 3072:
        det_pixels_train = det_ll_train_obs             # (N_train, 3072)
    else:
        raise ValueError(
            f"Unexpected det_ll_train_obs width {det_ll_train_obs.shape[1]} "
            "expected 3072 or 3092."
        )

    # ------------------------------------------------------------
    # 5) Align Theta to train split + pixels-only view
    # ------------------------------------------------------------
    if final_Theta_ll.shape[0] != len(train_indices):
        if final_Theta_ll.shape[0] == U_ll_base.shape[0]:
            final_Theta_ll = final_Theta_ll[train_indices]
        else:
            raise ValueError(
                f"Theta rows {final_Theta_ll.shape[0]} do not match "
                f"train size {len(train_indices)} or full N {U_ll_base.shape[0]}."
            )

    if final_Theta_ll.shape[1] != 3072:
        raise ValueError(f"Theta has {final_Theta_ll.shape[1]} cols, expected 3072 (pixels-only).")

    # ------------------------------------------------------------
    # 6) Reconstruct clean / worst-case pixels
    # ------------------------------------------------------------
    clean_recon_pixels = det_pixels_train + U_ll_train
    worst_case_pixels = clean_recon_pixels + final_Theta_ll

    # ------------------------------------------------------------
    # 7) Select samples
    # ------------------------------------------------------------
    num_samples_to_show = 4
    N_train = U_ll_train.shape[0]
    num_samples_to_show = min(num_samples_to_show, N_train)

    np.random.seed(fold_index + 42)
    sample_indices = np.random.choice(N_train, num_samples_to_show, replace=False)

    # ------------------------------------------------------------
    # 8) Reshape helpers
    # ------------------------------------------------------------
    def reshape_and_rescale_for_plot(pixel_vector_neg1_1):
        """3072 -> (32,32,3), rescale [-1,1] -> [0,1]."""
        if not isinstance(pixel_vector_neg1_1, torch.Tensor):
            pixel_vector_neg1_1 = torch.tensor(pixel_vector_neg1_1)
        pixel_vector_neg1_1 = pixel_vector_neg1_1.detach().cpu()

        if pixel_vector_neg1_1.numel() != 3072:
            return np.zeros((32, 32, 3))

        img_chw = pixel_vector_neg1_1.view(3, 32, 32)
        img_hwc = img_chw.permute(1, 2, 0).numpy()
        img_rescaled = (img_hwc + 1.0) / 2.0
        return np.clip(img_rescaled, 0.0, 1.0)

    def reshape_for_plot(pixel_vector):
        """3072 -> (32,32,3), keep raw values (for U / Θ heatmaps)."""
        if not isinstance(pixel_vector, torch.Tensor):
            pixel_vector = torch.tensor(pixel_vector)
        pixel_vector = pixel_vector.detach().cpu()
        if pixel_vector.numel() != 3072:
            return np.zeros((32, 32, 3))
        img_chw = pixel_vector.view(3, 32, 32)
        return img_chw.permute(1, 2, 0).numpy()

    # ------------------------------------------------------------
    # 9) Plot
    # ------------------------------------------------------------
    fig, axes = plt.subplots(num_samples_to_show, 5, figsize=(20, 4 * num_samples_to_show))
    if num_samples_to_show == 1:
        axes = np.array([axes])

    fig.suptitle(
        f"Visualizing Adversarial Perturbations (pixels-only)\n"
        f"{diroca_key_toplevel}, {fold_key_to_show}, ε={epsilon_run}, δ=0",
        fontsize=16, y=1.02
    )

    for row_i, idx in enumerate(sample_indices):
        # Col 1: Deterministic D
        ax = axes[row_i, 0]
        ax.imshow(reshape_and_rescale_for_plot(det_pixels_train[idx]))
        ax.set_title(f"Sample {idx}: D")
        ax.axis("off")

        # Col 2: U noise
        ax = axes[row_i, 1]
        img_u = reshape_for_plot(U_ll_train[idx])
        norm = np.max(np.abs(img_u)) if img_u.size > 0 else 0.1
        ax.imshow(img_u, cmap="RdBu_r", vmin=-norm-1e-6, vmax=norm+1e-6)
        ax.set_title(f"Sample {idx}: U")
        ax.axis("off")

        # Col 3: Θ adversary
        ax = axes[row_i, 2]
        img_th = reshape_for_plot(final_Theta_ll[idx])
        norm = np.max(np.abs(img_th)) if img_th.size > 0 else 0.1
        ax.imshow(img_th, cmap="RdBu_r", vmin=-norm-1e-6, vmax=norm+1e-6)
        ax.set_title(f"Sample {idx}: Θ")
        ax.axis("off")

        # Col 4: Clean D+U
        ax = axes[row_i, 3]
        ax.imshow(reshape_and_rescale_for_plot(clean_recon_pixels[idx]))
        ax.set_title(f"Sample {idx}: Clean (D+U)")
        ax.axis("off")

        # Col 5: Worst D+U+Θ
        ax = axes[row_i, 4]
        ax.imshow(reshape_and_rescale_for_plot(worst_case_pixels[idx]))
        ax.set_title(f"Sample {idx}: Worst (D+U+Θ)")
        ax.axis("off")

    plt.tight_layout(rect=[0, 0.03, 1, 0.98])
    plt.show()

    print("✓ Visualization completed successfully!")



==================== Visualizing Adversarial Perturbations (UPDATED) ====================


NameError: name 'all_results' is not defined

In [3]:
# === Helper: Huber-style pixel contamination (for evaluation) ===
import torch
import numpy as np

def apply_huber_contamination_cmnist(clean_data, alpha, noise_scale, noise_dims, seed=None, loc=0.0):
    """
    Contaminate ONLY the specified dimensions (columns) of clean_data with Gaussian noise.

    New setting reminder:
      - We will call this ONLY on LL pixel blocks during evaluation.
      - Digit/Color one-hots are NEVER passed here and never contaminated.

    Args:
        clean_data : (N, D) tensor or array
        alpha      : fraction of samples to contaminate in [0,1]
        noise_scale: std of Gaussian noise
        noise_dims : slice / list / np.ndarray / torch.Tensor of column indices
        seed       : RNG seed
        loc        : mean shift of noise (0.0 = zero-mean)
    """
    # --- Early exit (numerically safe) ---
    if alpha is None or noise_scale is None or np.isclose(alpha, 0.0) or np.isclose(noise_scale, 0.0):
        if isinstance(clean_data, torch.Tensor):
            return clean_data.clone().to(torch.float32)
        return torch.tensor(clean_data, dtype=torch.float32)

    # --- Convert input to tensor ---
    if isinstance(clean_data, torch.Tensor):
        data_cont = clean_data.to(torch.float32).clone()
    else:
        data_cont = torch.tensor(clean_data, dtype=torch.float32).clone()
    device = data_cont.device
    N, D = data_cont.shape

    # --- Build index tensor for noise_dims ---
    if isinstance(noise_dims, slice):
        start = 0 if noise_dims.start is None else noise_dims.start
        stop  = D if noise_dims.stop is None else noise_dims.stop
        step  = 1 if noise_dims.step is None else noise_dims.step
        noise_idx = torch.arange(start, stop, step, device=device)
    elif isinstance(noise_dims, (list, tuple, np.ndarray, torch.Tensor)):
        noise_idx = torch.as_tensor(noise_dims, dtype=torch.long, device=device)
    else:
        raise TypeError(f"Unsupported type for noise_dims: {type(noise_dims)}")

    # --- Keep only valid indices ---
    noise_idx = noise_idx[(noise_idx >= 0) & (noise_idx < D)]
    if noise_idx.numel() == 0:
        return data_cont

    # --- Extract sub-matrix to contaminate ---
    data_to_noise = data_cont.index_select(dim=1, index=noise_idx)  # (N, |noise_idx|)

    # --- Sample Gaussian noise ---
    rng = np.random.default_rng(seed)
    noise = rng.normal(loc=loc, scale=noise_scale, size=tuple(data_to_noise.shape)).astype(np.float32)
    noise_tensor = torch.tensor(noise, dtype=torch.float32, device=device)

    noisy_slice = data_to_noise + noise_tensor

    # --- Full contamination ---
    if alpha >= 1.0:
        data_cont[:, noise_idx] = noisy_slice
        return data_cont

    # --- Partial contamination ---
    n_contaminate = int(alpha * N)
    if n_contaminate <= 0:
        return data_cont

    idx_to_contaminate_np = rng.choice(N, size=n_contaminate, replace=False)
    idx_to_contaminate = torch.as_tensor(idx_to_contaminate_np, dtype=torch.long, device=device)

    # Replace only selected rows and selected columns
    data_cont.index_copy_(
        dim=0,
        index=idx_to_contaminate,
        source=data_cont.index_select(0, idx_to_contaminate).scatter(
            1,
            noise_idx.view(1, -1).expand(n_contaminate, -1),
            noisy_slice.index_select(0, idx_to_contaminate)
        )
    )
    return data_cont

print("✅ apply_huber_contamination_cmnist ready (pixels-only eval usage).")


✅ apply_huber_contamination_cmnist ready (pixels-only eval usage).


In [4]:
# === Helper: empirical abstraction error (pixels -> z & full) ===
import torch
import numpy as np

def calculate_empirical_error_flat(T_matrix, Dll_test_flat, Dhl_test):
    """
    Abstraction error (Frobenius MSE) in the NEW setting.

    NEW default:
      - T maps pixels -> z only.
      - LL labels (digit/color one-hots) are copied 1-1 to HL labels.
      - We evaluate against full HL vector [labels(20) | z(d_z)].

    Fallback:
      - If T is full (84 x 3092), use old full-linear evaluation.

    Args:
        T_matrix      : torch.Tensor or np.ndarray
                       Either (d_z, 3072) or (84, 3092)
        Dll_test_flat : (N, 3072) OR (N, 3092)  [pixels | LL labels]
        Dhl_test      : (N, 64)   OR (N, 84)    [HL labels | z]
    """
    try:
        # --- to tensors ---
        T_matrix = T_matrix if isinstance(T_matrix, torch.Tensor) else torch.tensor(T_matrix, dtype=torch.float32)
        Dll_test_flat = Dll_test_flat if isinstance(Dll_test_flat, torch.Tensor) else torch.tensor(Dll_test_flat, dtype=torch.float32)
        Dhl_test = Dhl_test if isinstance(Dhl_test, torch.Tensor) else torch.tensor(Dhl_test, dtype=torch.float32)

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        T_matrix = T_matrix.to(device)
        Dll_test_flat = Dll_test_flat.to(device)
        Dhl_test = Dhl_test.to(device)

        N = Dll_test_flat.shape[0]

        # ============================================================
        # Case A (NEW): T is z-only: (d_z, 3072)
        # ============================================================
        if T_matrix.shape[1] == 3072:
            d_z = T_matrix.shape[0]

            # Dll may be pixels-only or pixels+labels
            if Dll_test_flat.shape[1] == 3072:
                ll_pixels = Dll_test_flat
                ll_labels = None
            elif Dll_test_flat.shape[1] == 3092:
                ll_pixels = Dll_test_flat[:, :3072]
                ll_labels = Dll_test_flat[:, 3072:]  # (N,20)
            else:
                return float("inf")

            # Dhl may be z-only or labels+z
            if Dhl_test.shape[1] == d_z:
                hl_labels = ll_labels  # if present, else None
                hl_z = Dhl_test
            elif Dhl_test.shape[1] == 20 + d_z:
                hl_labels = Dhl_test[:, :20]
                hl_z = Dhl_test[:, 20:]
            else:
                return float("inf")

            with torch.no_grad():
                z_pred = ll_pixels @ T_matrix.T  # (N, d_z)

                if hl_labels is None:
                    # compare only z
                    diff = z_pred - hl_z
                else:
                    pred_full = torch.cat([hl_labels, z_pred], dim=1)
                    true_full = torch.cat([hl_labels, hl_z], dim=1)
                    diff = pred_full - true_full

                err = torch.norm(diff, p="fro") ** 2 / max(1, N)
            return float(err.item())

        # ============================================================
        # Case B (OLD/FULL): T is full map (84, 3092)
        # ============================================================
        if T_matrix.shape[1] == Dll_test_flat.shape[1] and T_matrix.shape[0] == Dhl_test.shape[1]:
            with torch.no_grad():
                Dhl_pred = Dll_test_flat @ T_matrix.T
                diff = Dhl_pred - Dhl_test
                err = torch.norm(diff, p="fro") ** 2 / max(1, N)
            return float(err.item())

        # If nothing matched, shape mismatch
        return float("inf")

    except Exception as e:
        print(f"Error in calculate_empirical_error_flat: {e}")
        return float("inf")


In [5]:
# === Evaluation: Clean & Huber noise (Cases 1 & 2) ===
import torch
import numpy as np
import pandas as pd
import os
import gc

print("\n--- Starting Evaluation: Clean & Huber-Noise (Cases 1 & 2) ---")

N_TRIALS = 5
NOISE_SCALE_FOR_ALPHA1 = 0.5   # noise level for alpha=1
ALPHA_VALUES_TO_TEST = [0.0, 1.0]  # 0 -> clean, 1 -> fully noisy

evaluation_records = []

# Count total configs for progress bar
total_methods_trained = 0
for method_group_key, method_data_inner in all_results.items():
    if isinstance(method_data_inner, dict):
        for fold_key, fold_data in method_data_inner.items():
            if isinstance(fold_data, dict) and 'error' not in fold_data:
                total_methods_trained += len(fold_data)

total_configs = total_methods_trained * len(ALPHA_VALUES_TO_TEST) * N_TRIALS

if total_configs == 0:
    print("Error: No valid training results found in 'all_results'.")
else:
    pbar_eval = tqdm(total=total_configs, desc="Evaluating (clean + huber)")

    LL_PIXEL_DIMS = slice(0, 3072)  # flattened RGB pixels

    for method_group_key, method_results_inner in all_results.items():
        if not isinstance(method_results_inner, dict):
            continue

        for fold_key, fold_data in method_results_inner.items():
            if not fold_key.startswith('fold_'):
                continue
            if 'error' in fold_data:
                continue

            fold_idx = int(fold_key.split('_')[-1])

            for run_key, run_result in fold_data.items():
                if 'error' in run_result or run_result.get('T_matrix') is None:
                    pbar_eval.update(len(ALPHA_VALUES_TO_TEST) * N_TRIALS)
                    continue

                T_matrix = run_result['T_matrix']
                test_idx = run_result['test_indices']
                if test_idx is None:
                    pbar_eval.update(len(ALPHA_VALUES_TO_TEST) * N_TRIALS)
                    continue

                # Human-readable method name
                if method_group_key.startswith('diroca_'):
                    eval_method_name = f"DiRoCA ({run_key})"
                elif method_group_key == 'gradca':
                    eval_method_name = "GradCA"
                elif method_group_key == 'baryca':
                    eval_method_name = "BaryCA"
                elif method_group_key == 'abslingam':
                    eval_method_name = run_key
                else:
                    eval_method_name = f"{method_group_key}_{run_key}"

                for alpha in ALPHA_VALUES_TO_TEST:
                    noise_scale = 0.0 if np.isclose(alpha, 0.0) else NOISE_SCALE_FOR_ALPHA1
                    loc_ll = 0.0  # label-preserving pixel noise (mean 0)

                    for trial in range(N_TRIALS):
                        trial_errors = []

                        for iota, eta in list(omega.items()):
                            try:
                                if iota not in Dll_samples or eta not in Dhl_samples:
                                    continue

                                ll_images_01, _, ll_digits, ll_colors = Dll_samples[iota]
                                max_idx = max(test_idx) if len(test_idx) > 0 else -1
                                if max_idx >= len(ll_images_01):
                                    continue

                                # --- Test split ---
                                ll_images_test_01 = ll_images_01[test_idx]      # (N_test,3,32,32) in [0,1]
                                ll_digits_test    = ll_digits[test_idx]
                                ll_colors_test    = ll_colors[test_idx]
                                Dhl_test_clean    = Dhl_samples[eta][test_idx]  # (N_test,84) clean HL

                                seed = hash((fold_idx, run_key, float(alpha),
                                             float(noise_scale), trial, str(iota))) % (2**32)

                                # Rescale pixels to tanh space [-1,1]
                                ll_images_test_tanh = ll_images_test_01 * 2.0 - 1.0

                                # flatten images and contaminate only pixels
                                ll_images_test_flat = ll_images_test_tanh.view(ll_images_test_tanh.shape[0], -1)
                                ll_images_cont_flat = apply_huber_contamination_cmnist(
                                    ll_images_test_flat, alpha, noise_scale,
                                    noise_dims=LL_PIXEL_DIMS, seed=seed, loc=loc_ll
                                )

                                # one-hot digits & colors (never contaminated)
                                ll_digits_onehot = F.one_hot(ll_digits_test, num_classes=10).float()
                                ll_colors_onehot = F.one_hot(ll_colors_test, num_classes=10).float()

                                device = ll_images_cont_flat.device
                                Dll_test_cont_flat_full = torch.cat(
                                    [ll_images_cont_flat,
                                     ll_digits_onehot.to(device),
                                     ll_colors_onehot.to(device)],
                                    dim=1
                                )  # (N_test, 3092)

                                # High-level stays clean (labels + z)
                                Dhl_test = Dhl_test_clean.to(device)

                                error = calculate_empirical_error_flat(
                                    T_matrix, Dll_test_cont_flat_full, Dhl_test
                                )
                                if not np.isnan(error) and error != float('inf'):
                                    trial_errors.append(error)

                            except Exception as e:
                                print(f"ERROR inner loop: {e} | "
                                      f"Context: M{eval_method_name}, F{fold_idx}, "
                                      f"R{run_key}, A{alpha}, N{noise_scale}, "
                                      f"T{trial}, Iota{iota}")
                                trial_errors.append(np.nan)

                        record = {
                            'shift_type': 'huber_noise',
                            'method': eval_method_name,
                            'fold': fold_idx,
                            'alpha': float(alpha),
                            'noise_scale': float(noise_scale),
                            'trial': trial,
                            'error': float(np.nanmean(trial_errors)) if trial_errors else np.nan,
                        }
                        if method_group_key.startswith('diroca_'):
                            record['train_epsilon'] = run_result.get('epsilon', np.nan)
                            record['train_delta']   = run_result.get('delta', np.nan)

                        evaluation_records.append(record)
                        pbar_eval.update(1)

                        # cleanup
                        del (ll_images_test_01, ll_images_test_tanh, ll_digits_test, ll_colors_test,
                             Dhl_test_clean, ll_images_test_flat, ll_images_cont_flat,
                             ll_digits_onehot, ll_colors_onehot, Dll_test_cont_flat_full)
                        if 'error' in locals():
                            del error
                        if 'trial_errors' in locals():
                            trial_errors[:] = []
                            del trial_errors

    pbar_eval.close()

    full_results_df = pd.DataFrame(evaluation_records)
    eval_output_path = os.path.join(output_dir, "cmnist_eval_clean_and_huber.pkl")
    full_results_df.to_pickle(eval_output_path)
    print(f"\nEvaluation results saved to {eval_output_path}")



--- Starting Evaluation: Clean & Huber-Noise (Cases 1 & 2) ---


NameError: name 'all_results' is not defined

In [6]:
# === Helper: camera/augmentation shifts ===
import numpy as np
import torchvision.transforms as T
import torch

def apply_camera_shifts_cmnist(images_01, alpha, transform, seed=None):
    """
    Apply a camera/augmentation transform to a fraction alpha of images.

    INPUT:
        images_01 : (N, C, H, W) tensor in [0,1]  (as stored in Dll_samples)
    OUTPUT:
        images_tanh_aug : (N, C, H, W) in tanh space [-1,1]
                          matching the optimization pipeline.
    """
    # alpha = 0 → no augmentation, but STILL convert to tanh
    if alpha <= 0.0:
        return images_01 * 2.0 - 1.0

    images_aug = images_01.clone()
    N = images_01.shape[0]
    rng = np.random.default_rng(seed)
    n_aug = int(alpha * N)

    if n_aug == 0:
        return images_01 * 2.0 - 1.0

    idx_all = np.arange(N)
    idx_aug = rng.choice(idx_all, size=n_aug, replace=False)

    for j in idx_aug:
        # torchvision transforms expect input in [0,1]
        images_aug[j] = transform(images_aug[j])

    # Convert augmented images to tanh space [-1,1]
    images_tanh_aug = images_aug * 2.0 - 1.0
    return images_tanh_aug

camera_transform = T.Compose([
    T.RandomAffine(
        degrees=10,              # small rotation
        translate=(0.1, 0.1),    # small translation
        scale=(0.9, 1.1)         # mild zoom
    ),
    T.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.1
    ),
    # Optionally: T.GaussianBlur(3, sigma=(0.1, 1.0)),
])

print("✅ apply_camera_shifts_cmnist & camera_transform ready.")


✅ apply_camera_shifts_cmnist & camera_transform ready.


In [7]:
# === Evaluation: Camera/Augmentation Shifts (Case 3) ===
import numpy as np
import pandas as pd
import torch

print("\n--- Starting Evaluation: Camera/Augmentation Shifts (Case 3) ---")

N_TRIALS = 5
ALPHA_VALUES_CAMERA = [0.0, 1.0]  # 0 -> clean, 1 -> all images augmented

eval_records_camera = []

if total_methods_trained == 0:
    print("Error: No valid training results found in 'all_results'.")
else:
    pbar_cam = tqdm(
        total=total_methods_trained * len(ALPHA_VALUES_CAMERA) * N_TRIALS,
        desc="Evaluating (camera shifts)"
    )

    for method_group_key, method_results_inner in all_results.items():
        if not isinstance(method_results_inner, dict):
            continue

        for fold_key, fold_data in method_results_inner.items():
            if not fold_key.startswith('fold_'):
                continue
            if 'error' in fold_data:
                continue

            fold_idx = int(fold_key.split('_')[-1])

            for run_key, run_result in fold_data.items():
                if 'error' in run_result or run_result.get('T_matrix') is None:
                    pbar_cam.update(len(ALPHA_VALUES_CAMERA) * N_TRIALS)
                    continue

                T_matrix = run_result['T_matrix']
                test_idx = run_result['test_indices']
                if test_idx is None:
                    pbar_cam.update(len(ALPHA_VALUES_CAMERA) * N_TRIALS)
                    continue

                # Human-readable method name
                if method_group_key.startswith('diroca_'):
                    eval_method_name = f"DiRoCA ({run_key})"
                elif method_group_key == 'gradca':
                    eval_method_name = "GradCA"
                elif method_group_key == 'baryca':
                    eval_method_name = "BaryCA"
                elif method_group_key == 'abslingam':
                    eval_method_name = run_key
                else:
                    eval_method_name = f"{method_group_key}_{run_key}"

                # Determine view
                if isinstance(T_matrix, torch.Tensor):
                    T_shape = tuple(T_matrix.shape)
                else:
                    T_shape = tuple(np.asarray(T_matrix).shape)

                is_opt_view  = (T_shape == (64, 3072))
                is_full_view = (T_shape == (84, 3092))

                if not (is_opt_view or is_full_view):
                    print(f"[Warning] Unexpected T shape {T_shape} for {eval_method_name}. Skipping.")
                    pbar_cam.update(len(ALPHA_VALUES_CAMERA) * N_TRIALS)
                    continue

                for alpha_cam in ALPHA_VALUES_CAMERA:
                    for trial in range(N_TRIALS):
                        trial_errors = []

                        for iota, eta in list(omega.items()):
                            try:
                                if iota not in Dll_samples or eta not in Dhl_samples:
                                    continue

                                ll_images, _, ll_digits, ll_colors = Dll_samples[iota]
                                max_idx = max(test_idx) if len(test_idx) > 0 else -1
                                if max_idx >= len(ll_images):
                                    continue

                                # LL images are in [0,1]
                                ll_images_test = ll_images[test_idx]          # (N,C,H,W) in [0,1]
                                ll_digits_test = ll_digits[test_idx]
                                ll_colors_test = ll_colors[test_idx]

                                # HL full vector, but we only want z-block
                                Dhl_test_full = Dhl_samples[eta][test_idx]   # (N,84)
                                Dhl_test_z    = Dhl_test_full[:, 20:]        # (N,64)

                                seed = hash((fold_idx, run_key, float(alpha_cam),
                                             trial, str(iota))) % (2**32)

                                # 1) apply camera shifts in [0,1] and convert to tanh
                                ll_images_shifted_tanh = apply_camera_shifts_cmnist(
                                    ll_images_test, alpha_cam, camera_transform, seed=seed
                                )  # (N,C,H,W) in [-1,1]

                                # 2) flatten pixels only (tanh space)
                                ll_pixels_shifted_flat = ll_images_shifted_tanh.view(
                                    ll_images_shifted_tanh.shape[0], -1
                                )  # (N,3072)

                                device = ll_pixels_shifted_flat.device

                                # Build LL input and T block depending on view
                                if is_opt_view:
                                    Dll_input = ll_pixels_shifted_flat        # (N,3072)
                                    T_use     = T_matrix                      # (64,3072)
                                    Dhl_target = Dhl_test_z.to(device)        # (N,64)
                                else:
                                    if not isinstance(T_matrix, torch.Tensor):
                                        T_matrix = torch.tensor(T_matrix, dtype=torch.float32)
                                    T_use = T_matrix[20:, :3072]             # (64,3072)

                                    Dll_input  = ll_pixels_shifted_flat      # (N,3072)
                                    Dhl_target = Dhl_test_z.to(device)       # (N,64)

                                # Compute error on z only
                                error = calculate_empirical_error_flat(
                                    T_use, Dll_input, Dhl_target
                                )
                                if not np.isnan(error) and error != float('inf'):
                                    trial_errors.append(error)

                            except Exception as e:
                                print(f"ERROR (camera) inner loop: {e} | "
                                      f"Context: M{eval_method_name}, F{fold_idx}, "
                                      f"R{run_key}, A{alpha_cam}, T{trial}, Iota{iota}")
                                trial_errors.append(np.nan)

                        record = {
                            'shift_type': 'camera_aug',
                            'method': eval_method_name,
                            'fold': fold_idx,
                            'alpha': float(alpha_cam),
                            'trial': trial,
                            'error': float(np.nanmean(trial_errors)) if trial_errors else np.nan,
                        }
                        if method_group_key.startswith('diroca_'):
                            record['train_epsilon'] = run_result.get('epsilon', np.nan)
                            record['train_delta']   = run_result.get('delta', np.nan)

                        eval_records_camera.append(record)
                        pbar_cam.update(1)

                        # cleanup
                        del (ll_images_test, ll_digits_test, ll_colors_test,
                             Dhl_test_full, Dhl_test_z,
                             ll_images_shifted_tanh, ll_pixels_shifted_flat)
                        if 'error' in locals():
                            del error
                        if 'trial_errors' in locals():
                            trial_errors[:] = []
                            del trial_errors

    pbar_cam.close()

    df_cam = pd.DataFrame(eval_records_camera)
    cam_output_path = os.path.join(output_dir, "cmnist_eval_camera_shifts.pkl")
    df_cam.to_pickle(cam_output_path)
    print(f"\nCamera-shift evaluation results saved to {cam_output_path}")



--- Starting Evaluation: Camera/Augmentation Shifts (Case 3) ---
Error: No valid training results found in 'all_results'.


In [8]:
# === Summary: camera-shift evaluation ===
import numpy as np

if 'df_cam' not in globals():
    raise ValueError("df_cam not found. Run the camera evaluation cell first.")

# Filter rows for clean (alpha=0) and fully shifted (alpha=1)
df_cam_clean = df_cam[
    (df_cam['shift_type'] == 'camera_aug') &
    (df_cam['alpha'].abs() < 1e-8)
]

df_cam_shift = df_cam[
    (df_cam['shift_type'] == 'camera_aug') &
    (df_cam['alpha'].round(3) == 1.0)
]

# Compute summaries
summary_cam_clean = (
    df_cam_clean.groupby('method')['error']
    .agg(['mean', 'std'])
    .sort_values('mean')
)

summary_cam_shift = (
    df_cam_shift.groupby('method')['error']
    .agg(['mean', 'std'])
    .sort_values('mean')
)

print("\n================ CAMERA SHIFT EVALUATION SUMMARY ================\n")

print("--- Camera: Clean (alpha = 0) ---")
if len(summary_cam_clean) == 0:
    print("No clean (alpha=0) results found.")
else:
    print(summary_cam_clean)

print("\n--- Camera: Fully Shifted (alpha = 1) ---")
if len(summary_cam_shift) == 0:
    print("No shifted (alpha=1) results found.")
else:
    print(summary_cam_shift)

print("\n===============================================================\n")


ValueError: df_cam not found. Run the camera evaluation cell first.

In [9]:
# === Visual sanity check: Huber contamination on pixels ===
import torch
import matplotlib.pyplot as plt

# pick observational environment explicitly
iota_obs = "obs" if "obs" in Dll_samples else list(Dll_samples.keys())[0]
ll_images, _, ll_digits, ll_colors = Dll_samples[iota_obs]

# pick some indices
idx = torch.arange(16)  # first 16 examples
imgs = ll_images[idx]   # (16, C, H, W)

# flatten for Huber (pixels only)
imgs_flat = imgs.view(imgs.shape[0], -1)
d = imgs_flat.shape[1]

imgs_huber_flat = apply_huber_contamination_cmnist(
    imgs_flat,
    alpha=1.0,
    noise_scale=0.5,
    noise_dims=slice(0, d),   # contaminate ONLY pixels
    seed=0,
    loc=0.0
)

imgs_huber = imgs_huber_flat.view_as(imgs)  # back to (N,C,H,W)

# plot original vs huber
n_show = 8
fig, axes = plt.subplots(2, n_show, figsize=(2*n_show, 4))

for j in range(n_show):
    # original
    ax = axes[0, j]
    img = imgs[j].detach().cpu()
    if img.shape[0] == 1:
        ax.imshow(img.squeeze(0).clamp(0, 1), cmap="gray")
    else:
        ax.imshow(img.permute(1, 2, 0).clamp(0, 1))
    ax.axis('off')
    ax.set_title(f"orig #{j}")

    # huber
    ax = axes[1, j]
    img_h = imgs_huber[j].detach().cpu()
    if img_h.shape[0] == 1:
        ax.imshow(img_h.squeeze(0).clamp(0, 1), cmap="gray")
    else:
        ax.imshow(img_h.permute(1, 2, 0).clamp(0, 1))
    ax.axis('off')
    ax.set_title(f"huber #{j}")

plt.tight_layout()
plt.show()


NameError: name 'Dll_samples' is not defined

In [10]:
# === Visual sanity check: camera/augmentation shifts ===
import matplotlib.pyplot as plt
import torch

# reuse imgs from previous cell (ll_images[idx])
imgs_cam = apply_camera_shifts_cmnist(
    imgs,
    alpha=1.0,              # apply to all 16
    transform=camera_transform,
    seed=0
)

n_show = 8
fig, axes = plt.subplots(2, n_show, figsize=(2*n_show, 4))

for j in range(n_show):
    # original
    ax = axes[0, j]
    img = imgs[j].detach().cpu()
    if img.shape[0] == 1:
        ax.imshow(img.squeeze(0).clamp(0, 1), cmap="gray")
    else:
        ax.imshow(img.permute(1, 2, 0).clamp(0, 1))
    ax.axis('off')
    ax.set_title(f"orig #{j}")

    # camera-shifted (note: in tanh space [-1,1])
    ax = axes[1, j]
    img_c = imgs_cam[j].detach().cpu()
    # for display, bring back to [0,1]
    img_c_disp = (img_c + 1.0) / 2.0
    if img_c_disp.shape[0] == 1:
        ax.imshow(img_c_disp.squeeze(0).clamp(0, 1), cmap="gray")
    else:
        ax.imshow(img_c_disp.permute(1, 2, 0).clamp(0, 1))
    ax.axis('off')
    ax.set_title(f"cam #{j}")

plt.tight_layout()
plt.show()


NameError: name 'imgs' is not defined

In [11]:
# === Norm statistics for each learned T (pixel->z) ===
import torch
import pandas as pd
import numpy as np

records = []

def extract_T_pix_to_z(T):
    """
    New setting expects T to be (d_z, 3072).
    If a full T (84, 3092) is passed, slice to z<-pixels part:
        rows 20: (z block), cols :3072 (pixels block).
    """
    T = T if isinstance(T, torch.Tensor) else torch.tensor(T, dtype=torch.float32)

    if T.ndim != 2:
        raise ValueError(f"T must be 2D, got shape {tuple(T.shape)}")

    # already pixel->z
    if T.shape[1] == 3072:
        return T

    # full old-style T (84,3092)
    if T.shape == (84, 3092):
        return T[20:, :3072]  # z rows, pixel cols

    # fallback: try to infer by matching pixel dim
    if T.shape[1] > 3072:
        return T[:, :3072]

    return T  # last resort

for method_group_key, folds in all_results.items():
    if not isinstance(folds, dict):
        continue

    for fold_key, fold_data in folds.items():
        if not fold_key.startswith("fold_"):
            continue
        if not isinstance(fold_data, dict) or "error" in fold_data:
            continue  # skip failed folds

        fold_idx = int(fold_key.split("_")[-1])

        for run_key, run_result in fold_data.items():
            if not isinstance(run_result, dict):
                continue
            if "error" in run_result or run_result.get("T_matrix") is None:
                continue

            T_raw = run_result["T_matrix"]
            try:
                T = extract_T_pix_to_z(T_raw)
            except Exception as e:
                print(f"[skip] could not parse T for {method_group_key}/{fold_key}/{run_key}: {e}")
                continue

            # method name logic (same as eval)
            if method_group_key.startswith("diroca_"):
                method_name = f"DiRoCA ({run_key})"
            elif method_group_key == "gradca":
                method_name = "GradCA"
            elif method_group_key == "baryca":
                method_name = "BaryCA"
            elif method_group_key == "abslingam":
                method_name = run_key
            else:
                method_name = f"{method_group_key}_{run_key}"

            # spectral & Frobenius norms on pixel->z map
            with torch.no_grad():
                fro_norm = torch.linalg.norm(T, ord='fro').item()
                try:
                    spec_norm = torch.linalg.norm(T, ord=2).item()
                except RuntimeError:
                    spec_norm = float("nan")

            # radii used in training (now stored at top-level)
            eps_train   = run_result.get("epsilon", np.nan)
            delta_train = run_result.get("delta", np.nan)

            records.append({
                "method_group": method_group_key,
                "run_key": run_key,
                "fold": fold_idx,
                "method": method_name,
                "T_shape": tuple(T.shape),
                "fro_norm": fro_norm,
                "spec_norm": spec_norm,
                "epsilon_train": float(eps_train) if eps_train is not None else np.nan,
                "delta_train": float(delta_train) if delta_train is not None else np.nan,
            })

norm_df = pd.DataFrame(records)

print("\n=== Per-run norms and training radii (first 50 rows) ===")
print(norm_df.head(50))

print("\n=== Aggregated by method (mean ± std) ===")
summary = norm_df.groupby("method")[["fro_norm", "spec_norm", "epsilon_train", "delta_train"]].agg(["mean", "std"])
print(summary)


NameError: name 'all_results' is not defined

In [12]:
# === Pixel-importance heatmaps + cheat index ===
import torch
import numpy as np
import matplotlib.pyplot as plt

def _as_tensor(x):
    return x if isinstance(x, torch.Tensor) else torch.tensor(x, dtype=torch.float32)

def compute_pixel_importance(T_matrix, img_shape=(3, 32, 32)):
    """
    T_matrix: (z_dim, 3072) mapping pixels -> z
    Returns:
      imp_maps: (z_dim, H, W) per-z heatmaps based on sum over RGB |weights|
      cheat_index: scalar concentration score (higher = more 'cheaty'/sparse)
    """
    T = _as_tensor(T_matrix).detach().cpu()
    z_dim, d_ll = T.shape
    assert d_ll == np.prod(img_shape), f"Expected d_ll={np.prod(img_shape)}, got {d_ll}"

    C, H, W = img_shape
    T_img = T.view(z_dim, C, H, W)
    imp_maps = T_img.abs().sum(dim=1)  # (z_dim, H, W)

    # cheat index: top-k / mean concentration averaged over z
    flat = imp_maps.view(z_dim, -1)  # (z_dim, H*W)
    k = min(50, flat.shape[1])
    topk_mean = flat.topk(k, dim=1).values.mean(dim=1)
    all_mean = flat.mean(dim=1) + 1e-12
    cheat_per_z = (topk_mean / all_mean)
    cheat_index = cheat_per_z.mean().item()

    return imp_maps.numpy(), cheat_index, cheat_per_z.numpy()

def plot_importance_maps(T_matrix, title="", n_show=8):
    imp_maps, cheat_index, cheat_per_z = compute_pixel_importance(T_matrix)
    z_dim, H, W = imp_maps.shape
    n_show = min(n_show, z_dim)

    fig, axes = plt.subplots(1, n_show, figsize=(3*n_show, 3))
    if n_show == 1:
        axes = [axes]

    for i in range(n_show):
        ax = axes[i]
        hm = imp_maps[i]
        vmax = np.max(hm) if np.max(hm) > 0 else 1.0
        ax.imshow(hm, cmap="magma", vmin=0.0, vmax=vmax)
        ax.set_title(f"z[{i}]  cheat={cheat_per_z[i]:.1f}")
        ax.axis("off")

    plt.suptitle(f"{title}\nCheat index (top50/mean avg): {cheat_index:.2f}", y=1.05)
    plt.tight_layout()
    plt.show()

    return cheat_index

# -------- Example usage --------
T_example = all_results['gradca']['fold_0']['gradca_run']['T_matrix']
cheat_val = plot_importance_maps(T_example, title="GradCA", n_show=8)
print("Cheat index:", cheat_val)


NameError: name 'all_results' is not defined

In [13]:
# === Semantic evaluation: interventional consistency (pixels -> z) ===
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

def _as_tensor_dev(x, device=None):
    t = x if isinstance(x, torch.Tensor) else torch.tensor(x, dtype=torch.float32)
    return t.to(device) if device is not None else t

@torch.no_grad()
def semantic_errors_pixels_to_z(
    T_matrix,
    det_ll_dict_opt,   # each (N, 3072) deterministic pixels only
    det_hl_dict_opt,   # each (N, 64) deterministic z only
    U_ll_hat_opt,      # (N, 3072) LL noise (test-time, maybe contaminated)
    U_hl_hat_opt,      # (N, 64)   HL noise (usually clean)
    omega,
    test_idx=None,     # optional index list/tensor for test split
    device=None,
):
    """
    Computes:
      e_{iota,eta}(T) = E || T (D_ll[iota]+U_ll) - (D_hl[eta]+U_hl) ||^2
    over all (iota, eta) in omega.

    Returns dict with per-pair errors plus:
      mean_iv_error, max_iv_error, var_iv_error
    """
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    T = _as_tensor_dev(T_matrix, device=device)
    U_ll = _as_tensor_dev(U_ll_hat_opt, device=device)
    U_hl = _as_tensor_dev(U_hl_hat_opt, device=device)

    if test_idx is not None:
        test_idx = torch.as_tensor(test_idx, dtype=torch.long, device=device)
        U_ll = U_ll.index_select(0, test_idx)
        U_hl = U_hl.index_select(0, test_idx)

    pair_errors = {}

    for iota, eta in omega.items():
        if iota not in det_ll_dict_opt or eta not in det_hl_dict_opt:
            continue

        Dll = _as_tensor_dev(det_ll_dict_opt[iota], device=device)
        Dhl = _as_tensor_dev(det_hl_dict_opt[eta], device=device)

        if test_idx is not None:
            Dll = Dll.index_select(0, test_idx)
            Dhl = Dhl.index_select(0, test_idx)

        # endo LL/HL (pixels->z setting)
        X_ll = Dll + U_ll           # (N_test, 3072)
        Z_hl = Dhl + U_hl           # (N_test, 64)

        Z_pred = X_ll @ T.T         # (N_test, 64)
        diff = Z_pred - Z_hl
        err = (diff.norm(p="fro")**2 / max(1, diff.shape[0])).item()

        pair_errors[(iota, eta)] = err

    errs = np.array(list(pair_errors.values()), dtype=np.float32)
    out = {
        "pair_errors": pair_errors,
        "mean_iv_error": float(np.mean(errs)) if len(errs) else np.nan,
        "max_iv_error": float(np.max(errs)) if len(errs) else np.nan,
        "var_iv_error": float(np.var(errs)) if len(errs) else np.nan,
    }
    return out


def run_semantic_eval_all_methods(
    all_results,
    det_ll_dict_opt, det_hl_dict_opt,
    U_ll_hat_opt, U_hl_hat_opt,
    omega
):
    """
    Drop-in evaluator over your all_results structure.
    Returns a DataFrame with mean/max/var interventional errors per run.
    """
    records = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for method_group_key, folds in all_results.items():
        if not isinstance(folds, dict):
            continue

        for fold_key, fold_data in folds.items():
            if not fold_key.startswith("fold_"):
                continue
            if not isinstance(fold_data, dict) or "error" in fold_data:
                continue
            fold_idx = int(fold_key.split("_")[-1])

            for run_key, run_result in fold_data.items():
                if not isinstance(run_result, dict) or "error" in run_result:
                    continue
                if run_result.get("T_matrix") is None:
                    continue

                T = run_result["T_matrix"]
                test_idx = run_result.get("test_indices", None)

                # match your naming conventions
                if method_group_key.startswith("diroca_"):
                    method_name = f"DiRoCA ({run_key})"
                elif method_group_key == "gradca":
                    method_name = "GradCA"
                elif method_group_key == "baryca":
                    method_name = "BaryCA"
                elif method_group_key == "abslingam":
                    method_name = run_key
                else:
                    method_name = f"{method_group_key}_{run_key}"

                sem = semantic_errors_pixels_to_z(
                    T, det_ll_dict_opt, det_hl_dict_opt,
                    U_ll_hat_opt, U_hl_hat_opt,
                    omega, test_idx=test_idx, device=device
                )

                records.append({
                    "method_group": method_group_key,
                    "run_key": run_key,
                    "fold": fold_idx,
                    "method": method_name,
                    "mean_iv_error": sem["mean_iv_error"],
                    "max_iv_error": sem["max_iv_error"],
                    "var_iv_error": sem["var_iv_error"],
                })

    return pd.DataFrame(records)

# -------- Example usage --------
# In this eval notebook, just use the opt-view noises:
U_ll_hat_run = U_ll_hat_opt
U_hl_hat_run = U_hl_hat_opt

sem_df = run_semantic_eval_all_methods(
    all_results,
    det_ll_dict_opt, det_hl_dict_opt,
    U_ll_hat_run, U_hl_hat_run,
    omega
)

print(sem_df.sort_values("mean_iv_error").head(20))
print(
    sem_df.groupby("method")[["mean_iv_error","max_iv_error","var_iv_error"]]
    .agg(["mean","std"])
)


NameError: name 'U_ll_hat_opt' is not defined